# Huber Loss en profundidad: precisión normal sin obedecer a los outliers

En [`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb) vimos que MSE
castiga fuerte los errores grandes y que MAE castiga todos los errores por
igual (ambas se estudian a fondo en
[`04_mse_en_profundidad.ipynb`](04_mse_en_profundidad.ipynb) y
[`05_mae_en_profundidad.ipynb`](05_mae_en_profundidad.ipynb)). Huber Loss es
útil cuando la mayoría de tus datos son razonables, pero unos pocos valores
extremos son errores de medición, errores de digitación o situaciones tan
raras que no deberían dictar el comportamiento normal del modelo.

La pregunta que responderemos aquí es muy concreta: **¿cómo puede Huber
mantener una recta precisa para los casos habituales sin ignorar por completo
los casos extraños?**

In [1]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from scipy.special import huber
from sklearn.datasets import load_diabetes
from sklearn.linear_model import HuberRegressor, LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Recordatorio visual: tres formas de castigar un error

Un **residuo** es $r = y - \hat y$: cuánto se alejó la predicción del valor
real. Ya viste esta gráfica en el notebook anterior; aquí la volvemos a mirar
porque es la clave de todo lo que sigue.

In [2]:
delta = 5.0
r = np.linspace(-14, 14, 400)
comparacion = pl.DataFrame({
    "residuo": np.concatenate([r, r, r]),
    "penalizacion": np.concatenate([r ** 2 / 2, np.abs(r), huber(delta, r)]),
    "perdida": ["MSE (r²/2)"] * len(r) + ["MAE (|r|)"] * len(r) + [f"Huber (δ={delta:.0f})"] * len(r),
})
px.line(
    comparacion, x="residuo", y="penalizacion", color="perdida",
    title="MSE crece cuadráticamente, MAE crece igual siempre, Huber empieza como MSE y termina como MAE",
).show()

| Pérdida | Fórmula para un residuo $r$ | Reacción a un error muy grande |
| --- | --- | --- |
| MAE | $\lvert r \rvert$ | Crece de forma lineal. Es robusta, pero no distingue tanto entre errores moderados y grandes. |
| MSE | $r^2$ | Crece cuadráticamente. Un outlier puede dominar la recta. |
| Huber | cuadrática cerca de 0; lineal lejos de 0 | Combina la precisión local de MSE con la robustez de MAE. |

No existe una pérdida "mejor" en todos los problemas: la elección depende de
qué tan grave permites que sea un error atípico durante el entrenamiento.

## 2. La fórmula y el umbral $\delta$

Huber usa un punto de cambio llamado $\delta$ (delta) que separa dos
"regímenes":

$$L_\delta(r) = \begin{cases}
\frac{1}{2}r^2 & \text{si } |r| \leq \delta \\
\delta\left(|r|-\frac{1}{2}\delta\right) & \text{si } |r| > \delta
\end{cases}$$

Antes de leer la fórmula por partes, míralo directamente en la curva: vamos a
sombrear cada región para que sea imposible confundirlas.

In [3]:
fig = go.Figure()
r_dentro = r[np.abs(r) <= delta]
r_fuera_izq = r[r < -delta]
r_fuera_der = r[r > delta]

fig.add_trace(go.Scatter(
    x=r_dentro, y=huber(delta, r_dentro), mode="lines", name=f"|r| ≤ δ: tramo cuadrático",
    line={"color": "#1f77b4", "width": 4},
))
fig.add_trace(go.Scatter(
    x=np.concatenate([r_fuera_izq, [np.nan], r_fuera_der]),
    y=np.concatenate([huber(delta, r_fuera_izq), [np.nan], huber(delta, r_fuera_der)]),
    mode="lines", name="|r| > δ: tramo lineal", line={"color": "#d62728", "width": 4},
))
fig.add_vline(x=delta, line_dash="dash", annotation_text="δ")
fig.add_vline(x=-delta, line_dash="dash", annotation_text="-δ")
fig.update_layout(title="Dentro de [-δ, δ] Huber es una parábola; fuera, es una línea recta")
fig.show()

Lee la gráfica de izquierda a derecha: **dentro** de la franja $[-\delta,
\delta]$ (azul), la curva es exactamente la misma parábola que MSE — pequeños
ajustes siguen importando y la recta queda afinada con precisión. **Fuera** de
esa franja (rojo), la curva se vuelve una línea recta, igual que MAE — el
castigo deja de crecer al cuadrado, así que un residuo de 20 no pesa mucho más
que uno de 10.

Elegir $\delta$ exige conocer la escala de tu variable: si predices precios en
dólares, $\delta$ se mide en dólares; si predices minutos, en minutos. Es,
literalmente, la respuesta a la pregunta "¿a partir de qué tamaño de error ya
no considero esto un caso normal?".

En este notebook usamos `scipy.special.huber` para calcular la pérdida por
residuo. Para entrenar un modelo, usamos `sklearn.linear_model.HuberRegressor`,
que implementa la optimización robusta — no escribimos un optimizador propio.

### Antes de usar `QuantileRegressor`: ¿qué es "la mediana" de un ajuste?

Más adelante compararemos Huber contra un modelo que minimiza MAE:
`QuantileRegressor(quantile=0.5)`. Ese `0.5` significa que el modelo intenta
predecir la **mediana condicional** en vez del promedio. Antes de usarlo, vale
la pena recordar con una gráfica simple por qué la mediana resiste tanto a los
valores extremos.

Imagina 9 sueldos normales y uno absurdamente alto (un error de captura). El
promedio se deja arrastrar por ese valor; la mediana (el valor de en medio,
una vez ordenados todos) casi ni se entera.

In [4]:
sueldos = pl.DataFrame({"sueldo": [2100, 2300, 2450, 2500, 2600, 2650, 2700, 2800, 2900, 95000]})
promedio = sueldos["sueldo"].mean()
mediana = sueldos["sueldo"].median()

fig = px.histogram(sueldos, x="sueldo", nbins=20, title="Un solo valor absurdo desplaza mucho el promedio, casi nada la mediana")
fig.add_vline(x=promedio, line_color="#d62728", annotation_text=f"promedio = {promedio:,.0f}")
fig.add_vline(x=mediana, line_color="#1f77b4", annotation_text=f"mediana = {mediana:,.0f}")
fig.show()

Ese es el mecanismo que hace robusto a `QuantileRegressor(quantile=0.5)`: en
vez de intentar que la suma de residuos al cuadrado sea mínima (lo que arrastra
la recta hacia los valores extremos, igual que el promedio), busca que la
mitad de los residuos queden por encima y la mitad por debajo — la definición
misma de mediana, ahora aplicada a una recta en vez de a un solo número.

## 3. Caso práctico: predicción de alquiler mensual

Una inmobiliaria quiere estimar el alquiler mensual a partir de los metros
cuadrados. La mayoría de sus registros son correctos. Sin embargo, alguien
escribió **12 000** en lugar de **1 200** para un departamento de 55 m².

El objetivo es estimar el alquiler habitual de nuevos departamentos, no repetir
ese error de digitación. Esta es una situación donde Huber suele ser mejor que
MSE: MSE intentará acomodar el valor 12 000; Huber limitará su influencia. MAE
también será robusta, pero Huber puede ajustar más finamente la mayoría de
errores pequeños.

In [5]:
entrenamiento = pl.DataFrame({
    "metros_cuadrados": [35, 42, 48, 55, 62, 70, 78, 86, 95, 55],
    "alquiler_usd": [820, 960, 1080, 1210, 1360, 1530, 1710, 1880, 2050, 12000],
    "registro": ["normal"] * 9 + ["error de digitación"],
})
entrenamiento

metros_cuadrados,alquiler_usd,registro
i64,i64,str
35,820,"""normal"""
42,960,"""normal"""
48,1080,"""normal"""
55,1210,"""normal"""
62,1360,"""normal"""
70,1530,"""normal"""
78,1710,"""normal"""
86,1880,"""normal"""
95,2050,"""normal"""


## 4. Entrenamos tres modelos sobre los mismos datos

- `LinearRegression` minimiza MSE.
- `QuantileRegressor(quantile=0.5)` minimiza una pérdida equivalente a MAE;
  predice la mediana condicional, como acabamos de ver.
- `HuberRegressor` usa Huber Loss. Su parámetro `epsilon` cumple un papel de
  tolerancia al outlier, pero trabaja sobre una escala interna robusta; no es
  exactamente el mismo $\delta$ en dólares de la gráfica anterior.

Los datos se manipulan con Polars; se convierten a NumPy solo al pasarlos a las
APIs de scikit-learn.

In [6]:
X_train = entrenamiento.select("metros_cuadrados").to_numpy()
y_train = entrenamiento["alquiler_usd"].to_numpy()

modelo_mse = LinearRegression().fit(X_train, y_train)
modelo_mae = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(X_train, y_train)
modelo_huber = HuberRegressor().fit(X_train, y_train.astype(float))

metros_linea = np.linspace(30, 100, 200).reshape(-1, 1)
fig = px.scatter(
    entrenamiento, x="metros_cuadrados", y="alquiler_usd", color="registro",
    title="La recta MSE es arrastrada por el error; Huber y MAE protegen el patrón normal",
    color_discrete_map={"normal": "black", "error de digitación": "#d62728"},
)
fig.add_trace(go.Scatter(x=metros_linea.ravel(), y=modelo_mse.predict(metros_linea), mode="lines", name="MSE (LinearRegression)"))
fig.add_trace(go.Scatter(x=metros_linea.ravel(), y=modelo_mae.predict(metros_linea), mode="lines", name="MAE (QuantileRegressor)"))
fig.add_trace(go.Scatter(x=metros_linea.ravel(), y=modelo_huber.predict(metros_linea), mode="lines", name="Huber (HuberRegressor)"))
fig.show()

## 5. ¿Cuál generaliza mejor a departamentos normales?

Evaluamos los tres modelos sobre departamentos nuevos y correctamente
registrados. Esta separación es crucial: evaluar con los mismos datos
corruptos de entrenamiento premiaría injustamente al modelo que persigue el
error de digitación.

In [7]:
prueba_limpia = pl.DataFrame({
    "metros_cuadrados": [38, 50, 60, 72, 84, 98],
    "alquiler_usd": [880, 1120, 1330, 1580, 1850, 2160],
})
X_test = prueba_limpia.select("metros_cuadrados").to_numpy()
y_test = prueba_limpia["alquiler_usd"].to_numpy()

resultados = pl.DataFrame([
    {
        "modelo": nombre,
        "MAE en prueba limpia": mean_absolute_error(y_test, modelo.predict(X_test)),
        "RMSE en prueba limpia": root_mean_squared_error(y_test, modelo.predict(X_test)),
    }
    for nombre, modelo in [("MSE", modelo_mse), ("MAE", modelo_mae), ("Huber", modelo_huber)]
]).sort("RMSE en prueba limpia")
resultados

modelo,MAE en prueba limpia,RMSE en prueba limpia
str,f64,f64
"""Huber""",12.721718,20.185275
"""MAE""",14.08805,22.567226
"""MSE""",958.702855,1081.023435


En esta prueba limpia, Huber y MAE deberían quedar cerca del mejor resultado
porque aprenden la tendencia de los nueve registros normales y reducen el
efecto del registro absurdo. MSE suele quedar peor porque su recta intentó
reducir un error de miles de dólares a costa de desviarse de los demás
departamentos.

Esto no significa que debamos ocultar todos los outliers. Primero hay que
investigarlos: quizá el departamento de 12 000 USD sea un penthouse real y
necesite variables nuevas como barrio, lujo o número de habitaciones. Huber es
apropiada aquí porque **sabemos** que el valor fue un error de digitación.

## 6. Guía corta para elegir

- Elige **MSE/RMSE** si los errores grandes son genuinamente más dañinos y
  deben mandar en la optimización (por ejemplo, cuando un error enorme tiene un
  costo desproporcionado: multas, seguridad, pérdida grave de clientes).
- Elige **MAE** si quieres medir el error típico en unidades claras y tratar
  cada minuto, dólar o unidad de error de forma proporcional.
- Prueba **Huber** cuando los outliers son pocos, no representan la población
  que quieres predecir, y todavía quieres que los errores pequeños se ajusten
  con precisión cuadrática.

No elijas una pérdida solo porque produzca el número más bajo en
entrenamiento. Define primero qué errores importan en el mundo real, usa un
conjunto de prueba representativo y compara las métricas que correspondan a
esa decisión.

## 7. Ejemplo con un dataset real: progresión de diabetes

Ahora usamos `load_diabetes`, un dataset de prueba real incluido en
scikit-learn. Contiene mediciones clínicas de 442 personas y una variable
objetivo continua: una medida de progresión de la enfermedad un año después.

El dataset no tiene errores extremos de digitación conocidos. Para estudiar
Huber de forma controlada, haremos algo parecido a un incidente común en
proyectos reales: **contaminaremos solo 18 etiquetas del entrenamiento** con un
error artificial de +500. La prueba se conserva limpia, así sabemos que el
modelo correcto es el que aprende de la mayoría de registros sanos y no el que
memoriza las etiquetas dañadas.

In [8]:
datos_diabetes = load_diabetes()
# Elegimos solo BMI para poder representar cada modelo como una recta.
indice_bmi = list(datos_diabetes.feature_names).index("bmi")
X = datos_diabetes.data[:, [indice_bmi]]
y_diabetes = datos_diabetes.target

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y_diabetes, test_size=0.3, random_state=42)

rng = np.random.default_rng(42)
indices_outlier = rng.choice(len(y_entrenamiento), size=18, replace=False)
y_entrenamiento_contaminado = y_entrenamiento.copy()
y_entrenamiento_contaminado[indices_outlier] += 500

print(f"Observaciones de entrenamiento: {len(y_entrenamiento)}; contaminadas: {len(indices_outlier)}")

Observaciones de entrenamiento: 309; contaminadas: 18


### Tres modelos, misma información

Para poder dibujar cada modelo como una recta, aquí usamos únicamente `bmi`
como variable de entrada. Los tres reciben exactamente la misma información y
las mismas etiquetas contaminadas. Usamos `StandardScaler` dentro de un
`Pipeline`: estandariza la variable usando únicamente el entrenamiento y evita
fuga de información hacia la prueba.

In [9]:
modelos_diabetes = {
    "MSE: LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "MAE: QuantileRegressor": make_pipeline(StandardScaler(), QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")),
    "Huber: HuberRegressor": make_pipeline(StandardScaler(), HuberRegressor()),
}
for modelo in modelos_diabetes.values():
    modelo.fit(X_entrenamiento, y_entrenamiento_contaminado)

resultados_diabetes = pl.DataFrame([
    {
        "modelo": nombre,
        "MAE en prueba limpia": mean_absolute_error(y_prueba, modelo.predict(X_prueba)),
        "RMSE en prueba limpia": root_mean_squared_error(y_prueba, modelo.predict(X_prueba)),
    }
    for nombre, modelo in modelos_diabetes.items()
]).sort("RMSE en prueba limpia")
resultados_diabetes

modelo,MAE en prueba limpia,RMSE en prueba limpia
str,f64,f64
"""Huber: HuberRegressor""",50.967712,62.367108
"""MAE: QuantileRegressor""",50.551278,63.01475
"""MSE: LinearRegression""",56.567838,67.357061


### Las rectas que aprende cada pérdida

La gráfica siguiente hace visible la diferencia. Los puntos rojos son las
etiquetas contaminadas que el modelo vio durante el entrenamiento. La recta de
MSE intenta acercarse a ellos porque sus errores al cuadrado pesan muchísimo.
La recta de Huber puede seguir más de cerca la nube de casos normales.

In [10]:
es_outlier = np.zeros(len(y_entrenamiento), dtype=bool)
es_outlier[indices_outlier] = True
puntos_entrenamiento = pl.DataFrame({
    "bmi": X_entrenamiento.ravel(),
    "progresion": y_entrenamiento_contaminado,
    "tipo": np.where(es_outlier, "etiqueta contaminada", "normal"),
})

bmi_linea = np.linspace(X_entrenamiento.min(), X_entrenamiento.max(), 200).reshape(-1, 1)
fig = px.scatter(
    puntos_entrenamiento, x="bmi", y="progresion", color="tipo",
    color_discrete_map={"normal": "black", "etiqueta contaminada": "#d62728"},
    opacity=0.6,
    title="MSE se deja arrastrar por las etiquetas contaminadas; Huber y MAE se resisten",
)
for nombre, modelo in modelos_diabetes.items():
    fig.add_trace(go.Scatter(x=bmi_linea.ravel(), y=modelo.predict(bmi_linea), mode="lines", name=nombre))
fig.show()

In [11]:
resultados_largos = resultados_diabetes.unpivot(
    index="modelo",
    on=["MAE en prueba limpia", "RMSE en prueba limpia"],
    variable_name="métrica",
    value_name="error",
)
px.bar(
    resultados_largos, x="modelo", y="error", color="métrica", barmode="group",
    title="Error en la prueba limpia: MSE queda más afectado por las 18 etiquetas dañadas",
).show()

### Qué demuestra este experimento

En la prueba limpia, Huber y MAE deberían mejorar claramente el RMSE frente a
MSE; el orden exacto puede cambiar según la muestra, la contaminación y el
valor de `epsilon`. MSE fue afectado por los errores de +500 y movió sus
parámetros para intentar reducirlos. Huber conserva información cuadrática de
los residuos normales mientras limita el efecto de los contaminados.

La lección no es "agrega outliers para que Huber gane". Es esta: cuando un
proceso real puede dañar una pequeña parte de las etiquetas y tu objetivo es
predecir nuevos casos normales, prueba un modelo robusto y evalúalo contra
datos de prueba que representen el uso real.

## 8. Ideas clave

- Huber Loss es cuadrática cerca de cero (como MSE) y lineal lejos de cero
  (como MAE); el umbral $\delta$ marca dónde cambia de comportamiento.
- Elegir $\delta$ requiere conocer qué error consideras "normal" en las
  unidades de tu variable.
- La mediana (y `QuantileRegressor(quantile=0.5)`) resiste a los outliers por
  la misma razón que Huber: no deja que un solo valor extremo domine el ajuste.
- Antes de aplicar un modelo robusto, investiga el outlier: si es un error de
  captura, protégete de él; si es un caso real, quizá necesites más variables
  para explicarlo en vez de ignorarlo.

**Ejercicio:** cambia el valor de `epsilon` al crear `HuberRegressor(epsilon=...)`
(el valor por defecto es 1.35) y vuelve a ejecutar la sección 4. Antes de mirar
el resultado, predice: ¿un `epsilon` más grande hará que Huber se parezca más a
MSE o más a MAE?